In [ ]:
# ==========================================
# Phase 2 - Cell 1: 原始数据加载与白名单提纯
# 输入: data/cs2_pro_detailed_RAW.csv
# 输出: df_filtered (白名单选手数据)
# ==========================================
import pandas as pd

raw_csv = "data/cs2_pro_detailed_RAW.csv"
df = pd.read_csv(raw_csv)
print(f"📦 原始数据加载完毕！总基数人数: {len(df)}")

# 白名单：HLTV Top 30 核心战队 + 高关注度补充战队
custom_whitelist = [
    # --- Top 30 核心 ---
    "Team Vitality", "Vitality", "FURIA Esports", "FURIA", "MOUZ",
    "Falcons", "Team Falcons", "Natus Vincere", "NAVI", "Aurora Gaming", "Aurora",
    "PARIVISION", "Team Spirit", "Spirit", "The MongolZ", "Astralis",
    "FaZe Clan", "FaZe", "FUT Esports", "FUT", "G2 Esports", "G2",
    "3DMAX", "Legacy", "GamerLegion", "paiN Gaming", "paiN",
    "HEROIC", "Team Liquid", "Liquid", "B8",
    "Ninjas in Pyjamas", "NIP", "9z Team", "9z", "NRG",
    "Gentle Mates", "Monte", "TYLOO", "BC.Game",
    "Passion UA", "BetBoom Team", "BetBoom", "HOTU",
    # --- 高关注度补充 ---
    "100 Thieves", "100T",
    "Lynn Vision", "Lynn Vision Gaming",
    "MIBR", "Imperial Esports", "Imperial",
    "FlyQuest", "OG", "TDK", "fnatic", "EYEBALLERS"
]

df_filtered = df[df['Team'].notna() & df['Team'].isin(custom_whitelist)].copy()
print(f"✨ 白名单提纯: {len(df_filtered)} 名选手")
df_filtered[['Player', 'Team', 'DPI', 'Sensitivity', 'Resolution']].head(8)


In [ ]:
# ==========================================
# Phase 2 - Cell 2: 数值清洗 + 质检去重 → Master.csv
# 输入: df_filtered
# 输出: data/cs2_pro_2026_Active_Master.csv
# ==========================================
import pandas as pd
import numpy as np
import re

# 1. 聚焦关键列
desired_columns = [
    'Player', 'Team', 'DPI', 'Sensitivity', 'eDPI', 'Zoom Sensitivity',
    'Hz', 'Resolution', 'Aspect Ratio', 'Scaling Mode'
]
actual_columns = [col for col in desired_columns if col in df_filtered.columns]
df_clean = df_filtered[actual_columns].copy()

# 2. 数值化（处理 1,200 / 400dpi 等混合格式）
def clean_numeric_text(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace(',', '').replace(' ', '')
    match = re.search(r'[\d\.]+', val_str)
    return float(match.group()) if match else np.nan

numeric_cols = ['DPI', 'Sensitivity', 'eDPI', 'Zoom Sensitivity', 'Hz']
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_numeric_text)

# 3. 回填缺失 eDPI
if 'eDPI' in df_clean.columns:
    mask = df_clean['eDPI'].isna() & df_clean['DPI'].notna() & df_clean['Sensitivity'].notna()
    df_clean.loc[mask, 'eDPI'] = df_clean.loc[mask, 'DPI'] * df_clean.loc[mask, 'Sensitivity']

# 4. 拆解分辨率
if 'Resolution' in df_clean.columns:
    res_df = df_clean['Resolution'].astype(str).str.extract(r'(\d+)x(\d+)').astype(float)
    df_clean['Res_Width'] = res_df[0]
    df_clean['Res_Height'] = res_df[1]

# 5. 数据质检
before_qc = len(df_clean)
df_clean = df_clean.dropna(subset=['eDPI', 'Resolution']).copy()
df_clean = df_clean[df_clean['Player'].str.lower() != df_clean['Team'].str.lower()]
df_clean['Player_lower'] = df_clean['Player'].str.lower()
df_clean = df_clean.drop_duplicates(subset=['Player_lower'], keep='first')
df_clean = df_clean.drop(columns=['Player_lower'])

print(f"🛡️ 质检: {before_qc} → {len(df_clean)} 人 (剔除 {before_qc - len(df_clean)} 条脏数据)")

# 6. 导出
master_csv = "data/cs2_pro_2026_Active_Master.csv"
df_clean.to_csv(master_csv, index=False, encoding='utf-8-sig')
print(f"💾 已保存: {master_csv} ({len(df_clean.columns)} 列)")
df_clean.head(5)


In [ ]:
# ==========================================
# Phase 2 - Cell 3: 特征增强 (RAW 字段回填) → Final.csv
# 输入: df_clean + data/cs2_pro_detailed_RAW.csv
# 输出: data/cs2_pro_2026_Active_Final.csv
# ==========================================
import pandas as pd

# 从 RAW 表中提取高级画面/延迟特征
df_raw = pd.read_csv('data/cs2_pro_detailed_RAW.csv', low_memory=False)

target_columns = [
    'Player', 'Brightness', 'Display Mode', 'V-Sync', 'NVIDIA Reflex Low Latency',
    'NVIDIA G-Sync', 'Maximum FPS In Game', 'Model / Texture Detail',
    'Shader Detail', 'Particle Detail', 'Ambient Occlusion',
    'Global Shadow Quality', 'Multisampling Anti-Aliasing Mode', 'Texture Filtering Mode',
    'Refresh Rate', 'Follow Recoil', 'Red', 'Green', 'Blue',
    'FOV', 'Offset X', 'Offset Y', 'Offset Z',
    'Radar Centers The Player', 'Radar is Rotating', 'Radar Map Zoom',
    'HUD Scale', 'HUD Color', 'Windows Sensitivity'
]

# 去重缝合
available_cols = [c for c in target_columns if c in df_raw.columns]
df_features = df_raw[available_cols].drop_duplicates(subset=['Player'])
df_pro = pd.merge(df_clean, df_features, on='Player', how='left', suffixes=('', '_new'))

# 字段覆盖（去掉 merge 产生的 _new 后缀）
for col in available_cols:
    if col != 'Player' and col + '_new' in df_pro.columns:
        df_pro[col] = df_pro[col + '_new']
        df_pro = df_pro.drop(columns=[col + '_new'])

# Unknown 填充
fill_cols = ['Display Mode', 'V-Sync', 'NVIDIA Reflex Low Latency', 'NVIDIA G-Sync',
             'Model / Texture Detail', 'Shader Detail', 'Particle Detail', 'Ambient Occlusion',
             'Multisampling Anti-Aliasing Mode', 'Texture Filtering Mode',
             'Follow Recoil', 'HUD Color']
for col in fill_cols:
    if col in df_pro.columns:
        df_pro[col] = df_pro[col].fillna('Unknown')

# 强制数值化关键列（防止 CSV 回读后变字符串）
numeric_force = ['eDPI', 'Hz', 'DPI', 'Sensitivity', 'Red', 'Green', 'Blue',
                 'Offset X', 'Offset Y', 'Offset Z', 'FOV', 'Radar Map Zoom']
for col in numeric_force:
    if col in df_pro.columns:
        df_pro[col] = pd.to_numeric(df_pro[col], errors='coerce')

# 导出
final_csv = "data/cs2_pro_2026_Active_Final.csv"
df_pro.to_csv(final_csv, index=False, encoding='utf-8-sig')
print(f"💾 已保存: {final_csv}")
print(f"📊 {len(df_pro)} 人 × {len(df_pro.columns)} 维特征")
df_pro.head(5)


In [ ]:
# ==========================================
# Phase 2 - Cell 4: 数据就绪检查
# ==========================================
import pandas as pd

df_pro = pd.read_csv('data/cs2_pro_2026_Active_Final.csv')

print(f"✅ 最终数据集: {len(df_pro)} 名选手, {len(df_pro.columns)} 个维度")
print(f"🏢 战队数: {df_pro['Team'].nunique()}")

# 核心字段缺失检查
key_fields = ['eDPI', 'DPI', 'Sensitivity', 'Resolution', 'Hz']
missing = {f: df_pro[f].isna().sum() for f in key_fields if f in df_pro.columns}
if any(missing.values()):
    for f, cnt in missing.items():
        if cnt > 0:
            print(f"  ⚠️ {f}: {cnt} 条缺失")
else:
    print("✅ 核心字段完整，可进入 03_final_report 生成图表")
